In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
input_file = Path('/Users/miasmacbook/Desktop/6-months_project/database/combined.tsv')
output_dir = Path('/Users/miasmacbook/Desktop/6-months_project/data/Histogram')
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(input_file, sep='\t')
print(df.shape)
df.head()

In [ ]:
# Keep columns 
needed = ["diseaseFromSourceMappedId", "targetId"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Available columns: {list(df.columns)}")

df_pairs = df[needed].dropna().copy()

In [ ]:
pairs_nodup = (
    df_pairs
    .drop_duplicates(subset=needed)
    .sort_values(needed)
    .reset_index(drop=True)
)

table1_path = output_dir / "disease_target_nodup.csv"
pairs_nodup.to_csv(table1_path, index=False)

In [ ]:
uniq_counts = (
    pairs_nodup
    .groupby("diseaseFromSourceMappedId", as_index=False)["targetId"]
    .nunique()
    .rename(columns={"targetId": "n_unique_targetId"})
    .sort_values("n_unique_targetId", ascending=False)
    .reset_index(drop=True)
)

table2_path = output_dir / "disease_unique_target_counts.csv"
uniq_counts.to_csv(table2_path, index=False)

In [ ]:
freq = uniq_counts["n_unique_targetId"].astype(int).value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(freq.index, freq.values)
plt.xlabel("Number of unique genes (targetId) per disease")
plt.ylabel("Number of diseases (diseaseFromSourceMappedId)")
plt.title("Distribution of unique genes per disease")
plt.tight_layout()

# Save figure
fig_path = output_dir / "hist_diseases_by_gene_count.png"
plt.savefig(fig_path, dpi=200)
plt.show()

# --- 8) Quick sanity checks / previews ---
print("Raw rows:", df.shape[0])
print("Unique disease-gene pairs (Table 1):", pairs_nodup.shape[0])
print("Number of diseases (Table 2):", uniq_counts.shape[0])

display(pairs_nodup.head())
display(uniq_counts.head())

print("\nSaved:")
print("Table 1:", table1_path)
print("Table 2:", table2_path)
print("Figure :", fig_path)

In [ ]:
# Create figure/axes explicitly
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(freq_1_20.index, freq_1_20.values)

ax.set_xticks(range(1, 21))
ax.set_xlabel("Number of unique genes (targetId) per disease")
ax.set_ylabel("Number of diseases (diseaseFromSourceMappedId)")
ax.set_title("Distribution of unique genes per disease (1–20 genes)")

# Labels on bars
for b in bars:
    h = b.get_height()
    ax.text(
        b.get_x() + b.get_width()/2,
        h,
        str(int(h)),
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_ylim(0, freq_1_20.max() * 1.12)

fig.tight_layout()

# Save BEFORE show
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)